In [1]:
import os

from argparse import Namespace
import collections
import nltk.data
import numpy as np
import pandas as pd
import re
import string
# from tqdm import tqdm_notebook
# tqdm_notebook deprecated; adding the following
from tqdm.notebook import tqdm

In [2]:
# ADDED
from pathlib import Path
import os
cwd = Path.cwd()
while cwd.name != "PyTorchNLPBook":
    if cwd.parent == cwd: raise RutimeError("Couldn't find PyTorchNLPBook")
    cwd = cwd.parent
os.chdir(cwd)
Path.cwd()

WindowsPath('D:/Source/gh-pc/PyTorchNLPBook')

In [3]:
# ADDED: nltk punkt data are not included when nltk module was installed; they need to be downloaded once into the environment.
import nltk

try:
    nltk.data.find("tokenizers/punkt")
    print("tokenizers/punkt found")
except LookupError:
    print("downloading nltk punkt into environment")
    nltk.download("punkt")

try:
    nltk.data.find("tokenizers/punkt_tab")
    print("tokenizers/punkt_tab found")
except LookupError:
    print("downloading nltk punkt_tab into environment")
    nltk.download("punkt_tab")

tokenizers/punkt found
tokenizers/punkt_tab found


In [4]:
args = Namespace(
    raw_dataset_txt="data/books/frankenstein.txt",
    window_size=5,
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    output_munged_csv="data/books/frankenstein_with_splits.csv",
    seed=1337
)

In [5]:
# Split the raw text book into sentences
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
with open(args.raw_dataset_txt) as fp:
    book = fp.read()
sentences = tokenizer.tokenize(book)

In [6]:
print (len(sentences), "sentences")
print ("Sample:", sentences[100])

3427 sentences
Sample: No incidents have hitherto befallen us that would make a figure in a
letter.


In [7]:
# Clean sentences
def preprocess_text(text):
    text = ' '.join(word.lower() for word in text.split(" "))
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text

In [8]:
cleaned_sentences = [preprocess_text(sentence) for sentence in sentences]

In [9]:
# ADDED
print ("Sample pre-cleaned sentence:\n", sentences[0],"\n\n")
print ("Sample cleaned sentence:\n", cleaned_sentences[0])

Sample pre-cleaned sentence:
 Frankenstein,

or the Modern Prometheus


by

Mary Wollstonecraft (Godwin) Shelley


Letter 1


St. Petersburgh, Dec. 11th, 17--

TO Mrs. Saville, England

You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
forebodings. 


Sample cleaned sentence:
 frankenstein , or the modern prometheus by mary wollstonecraft godwin shelley letter st . petersburgh , dec . th , to mrs . saville , england you will rejoice to hear that no disaster has accompanied the commencement of an enterprise which you have regarded with such evil forebodings . 


In [10]:
# ADDED
list(w for w in cleaned_sentences[0].split(' ')[:6])

['frankenstein', ',', 'or', 'the', 'modern', 'prometheus']

In [11]:
# Global vars
MASK_TOKEN = "<MASK>"

#### Comments (added)
In the `windows = ...` line, a `(list[str], int)` tuple is created for each cleaned sentence. The string list element will have copies of `MASK_TOKEN` added at the start and end of the sentence. The integer will be `arg.window_size * 2 + 1`. For example, cleaned_sentences[100] is:

` no incidents have hitherto befallen us that would make a figure in a letter . `

The list element of the tuple will be:

`['<MASK>', '<MASK>', '<MASK>', '<MASK>', '<MASK>', 'no', 'incidents', ... , 'a', 'letter', '.', '', '<MASK>', '<MASK>', '<MASK>', '<MASK>', '<MASK>']`

The call to `nltk.ngrams()` will create ngrams of length `args.window_size * 2 + 1` as sliding windows over that padded string. For example, for `cleaned_sentences[0]`, the first derived n-gram will be:

`('<MASK>', '<MASK>', '<MASK>', '<MASK>', '<MASK>', 'frankenstein', ',', 'or', 'the', 'modern', 'prometheus')`

`windows` will be a flattened list of all such windowed sequences over each sentence for all sentences.

The `for window in ...` loop, for each sentence window token sequence, will identify the (n+1)th token (n = `arg.windows_size`), and form a string containing all of the tokens *except* the (n+1)th token, and skipping any of the mask tokens. For example, from the first n-gram for `cleaned_sentences[0]`, the string will be:

`", or the modern prometheus"`

It will then create a tuple with that string plus the target token:

`(", or the modern prometheus", "frankenstein")`

It creates a list of these across all of the n-gram sequences.


In [12]:
# Create windows
flatten = lambda outer_list: [item for inner_list in outer_list for item in inner_list]
windows = flatten([list(nltk.ngrams([MASK_TOKEN] * args.window_size + sentence.split(' ') + \
    [MASK_TOKEN] * args.window_size, args.window_size * 2 + 1)) \
    for sentence in tqdm(cleaned_sentences)])

# Create cbow data
data = []
for window in tqdm(windows):
    target_token = window[args.window_size]
    context = []
    for i, token in enumerate(window):
        if token == MASK_TOKEN or i == args.window_size:
            continue
        else:
            context.append(token)
    data.append([' '.join(token for token in context), target_token])
    
            
# Convert to dataframe
cbow_data = pd.DataFrame(data, columns=["context", "target"])

  0%|          | 0/3427 [00:00<?, ?it/s]

  0%|          | 0/90698 [00:00<?, ?it/s]

In [13]:
# Create split data
n = len(cbow_data)
def get_split(row_num):
    if row_num <= n*args.train_proportion:
        return 'train'
    elif (row_num > n*args.train_proportion) and (row_num <= n*args.train_proportion + n*args.val_proportion):
        return 'val'
    else:
        return 'test'
cbow_data['split']= cbow_data.apply(lambda row: get_split(row.name), axis=1)

In [14]:
cbow_data.head()

,context,target,split
0,", or the modern prometheus",frankenstein,train
1,frankenstein or the modern prometheus by,",",train
2,"frankenstein , the modern prometheus by mary",or,train
3,"frankenstein , or modern prometheus by mary wo...",the,train
4,"frankenstein , or the prometheus by mary wolls...",modern,train


In [15]:
# Write split data to file
cbow_data.to_csv(args.output_munged_csv, index=False)